In [1]:
from pathlib import Path
import sys
import pandas as pd

ROOT: Path = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
from eval_utils import LABEL_COLS, score_predictions

data: pd.DataFrame = pd.read_csv(ROOT / "data/processed/sbdh_splits.csv")
train: pd.DataFrame = data[data["split"] == "train"].reset_index(drop=True)
dev: pd.DataFrame = data[data["split"] == "dev"].reset_index(drop=True)
print(f"train: {len(train):,} | dev: {len(dev):,}")

# majority class per label, learned from train, applied to dev
majority: dict[str, int] = {col: int(train[col].mode()[0]) for col in LABEL_COLS}
dev_pred_majority: pd.DataFrame = pd.DataFrame({col: [majority[col]] * len(dev) for col in LABEL_COLS})

scores_majority: pd.DataFrame = score_predictions(dev[LABEL_COLS], dev_pred_majority, "majority")
print(majority)
print(scores_majority[["label", "macro_f1", "weighted_f1"]].round(3).to_string(index=False))

train: 4,803 | dev: 1,029
{'sdoh_community_present': 1, 'sdoh_community_absent': 0, 'sdoh_education': 0, 'sdoh_economics': 0, 'sdoh_environment': 1, 'behavior_alcohol': 3, 'behavior_tobacco': 3, 'behavior_drug': 0}
                 label  macro_f1  weighted_f1
sdoh_community_present     0.393        0.510
 sdoh_community_absent     0.467        0.818
        sdoh_education     0.493        0.956
        sdoh_economics     0.251        0.454
      sdoh_environment     0.259        0.493
      behavior_alcohol     0.107        0.194
      behavior_tobacco     0.099        0.163
         behavior_drug     0.155        0.493


In [2]:
import importlib, regex_baseline
importlib.reload(regex_baseline)
from eval_utils import per_class_report

SUB_COLS: list[str] = list(regex_baseline.SUBSTANCE_TERMS)
dev_pred_sub: pd.DataFrame = regex_baseline.predict_substances(dev["SOCIAL_HISTORY_TEXT"])

for col in SUB_COLS:
    print(f"\n=== {col} ===")
    print(per_class_report(dev[col], dev_pred_sub[col]))


=== behavior_alcohol ===
              precision    recall  f1-score   support

           0       0.91      0.72      0.80       267
           1       0.81      0.60      0.69       288
           2       0.33      0.57      0.42        56
           3       0.69      0.93      0.79       374
           4       0.00      0.00      0.00        44

    accuracy                           0.72      1029
   macro avg       0.55      0.57      0.54      1029
weighted avg       0.73      0.72      0.71      1029


=== behavior_tobacco ===
              precision    recall  f1-score   support

           0       0.89      0.85      0.87       197
           1       0.51      0.63      0.56       151
           2       0.82      0.73      0.77       281
           3       0.77      0.92      0.83       339
           4       0.00      0.00      0.00        61

    accuracy                           0.76      1029
   macro avg       0.60      0.62      0.61      1029
weighted avg       0.72  

In [ ]:
from sklearn.metrics import confusion_matrix
import re

def show_errors(label: str, true_cls: int, pred_cls: int, n: int = 15) -> None:
    pattern: str = regex_baseline.SUBSTANCE_TERMS[label]
    mask = (dev[label] == true_cls) & (dev_pred_sub[label] == pred_cls)
    print(f"{label}: true={true_cls} pred={pred_cls} -> {int(mask.sum())} notes\n")
    for text in dev.loc[mask, "SOCIAL_HISTORY_TEXT"].head(n):
        hits = [s.strip() for s in re.split(regex_baseline.SPLIT, text) if re.search(pattern, s, flags=re.IGNORECASE)]
        print(" | ".join(hits)[:300])
        print("-" * 60)

for col in SUB_COLS:
    print(col)
    print(confusion_matrix(dev[col], dev_pred_sub[col], labels=[0, 1, 2, 3, 4]))

# largest drug error group: labeled never/none, predicted present
show_errors("behavior_drug", true_cls=3, pred_cls=1)

In [7]:
importlib.reload(regex_baseline)

SDOH_COLS: list[str] = [c for c in LABEL_COLS if c.startswith("sdoh_")]
dev_pred_all: pd.DataFrame = regex_baseline.predict_all(dev["SOCIAL_HISTORY_TEXT"])

scores_regex: pd.DataFrame = score_predictions(dev[LABEL_COLS], dev_pred_all[LABEL_COLS], "regex")
compare: pd.DataFrame = scores_majority[["label", "macro_f1"]].merge(
    scores_regex[["label", "macro_f1"]], on="label", suffixes=("_majority", "_regex"))
print(compare.round(3).to_string(index=False))

for col in SDOH_COLS:
    print(f"\n{col}")
    print(confusion_matrix(dev[col], dev_pred_all[col], labels=sorted(dev[col].unique())))

                 label  macro_f1_majority  macro_f1_regex
sdoh_community_present              0.393           0.871
 sdoh_community_absent              0.467           0.795
        sdoh_education              0.493           0.830
        sdoh_economics              0.251           0.834
      sdoh_environment              0.259           0.765
      behavior_alcohol              0.107           0.541
      behavior_tobacco              0.099           0.607
         behavior_drug              0.155           0.522

sdoh_community_present
[[317  45]
 [ 79 588]]

sdoh_community_absent
[[815  86]
 [ 24 104]]

sdoh_education
[[972  27]
 [  1  29]]

sdoh_economics
[[567  23  31]
 [ 24 115   4]
 [ 30  22 213]]

sdoh_environment
[[321  42   0]
 [ 38 612   3]
 [  2   6   5]]


In [ ]:
def show_sdoh_errors(label: str, true_cls: int, pred_cls: int, n: int = 15) -> None:
    mask = (dev[label] == true_cls) & (dev_pred_all[label] == pred_cls)
    print(f"{label}: true={true_cls} pred={pred_cls} -> {int(mask.sum())} notes\n")
    for text in dev.loc[mask, "SOCIAL_HISTORY_TEXT"].head(n):
        print(regex_baseline.prep_text(text)[:300])
        print("-" * 60)

show_sdoh_errors("sdoh_community_present", true_cls=1, pred_cls=0)  # missed support
show_sdoh_errors("sdoh_community_absent", true_cls=0, pred_cls=1)   # false loss of support

In [8]:
test: pd.DataFrame = data[data["split"] == "test"].reset_index(drop=True)
test_pred_regex: pd.DataFrame = regex_baseline.predict_all(test["SOCIAL_HISTORY_TEXT"])

test_pred_majority: pd.DataFrame = pd.DataFrame({col: [majority[col]] * len(test) for col in LABEL_COLS})
test_scores: pd.DataFrame = pd.concat([
    score_predictions(test[LABEL_COLS], test_pred_majority[LABEL_COLS], "majority"),
    score_predictions(test[LABEL_COLS], test_pred_regex[LABEL_COLS], "regex"),
])

summary: pd.DataFrame = test_scores.pivot(index="label", columns="model", values="macro_f1").reindex(LABEL_COLS)
summary.loc["MEAN"] = summary.mean()
print(summary.round(3).to_string())

(ROOT / "outputs/metrics").mkdir(parents=True, exist_ok=True)
(ROOT / "outputs/predictions").mkdir(parents=True, exist_ok=True)
test_scores.to_csv(ROOT / "outputs/metrics/test_scores_regex_majority.csv", index=False)
pd.concat([test[["ROW_ID"]], test_pred_regex], axis=1).to_csv(ROOT / "outputs/predictions/test_pred_regex.csv", index=False)
print("saved metrics + predictions")

model                   majority  regex
label                                  
sdoh_community_present     0.394  0.884
sdoh_community_absent      0.473  0.785
sdoh_education             0.492  0.834
sdoh_economics             0.252  0.811
sdoh_environment           0.256  0.848
behavior_alcohol           0.101  0.580
behavior_tobacco           0.096  0.621
behavior_drug              0.154  0.529
MEAN                       0.277  0.737
saved metrics + predictions
